# Surface conversion test: nibabel vs `mris_convert --to-scanner`

Experimental check of whether the FreeSurfer container step in the `convert_fs_surface_to_stl` rule
can be replaced by a pure nibabel conversion. Nothing here is used by the workflow.

Steps:
1. Reference STLs from FreeSurfer 7.4.1 `mris_convert --to-scanner` (apptainer).
2. Candidate STLs from nibabel.
3. Geometric comparison.
4. Byte-level comparison of the binary STL records.
5. Optional: export a `.vtp` coloured by the normal differences for visual inspection.

Run with the gonzo pixi environment (e.g. `pixi run jupyter lab`).

In [1]:
import subprocess
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import pyvista as pv
from scipy.spatial import cKDTree

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists())
SUBJECT = "sub-01"
SURF_DIR = ROOT / f"mri_processed_data/fastsurfer/{SUBJECT}/surf"
ORIG_MGZ = SURF_DIR.parent / "mri/orig.mgz"
LICENSE = ROOT / "docker/license.txt"
SIF = ROOT / "freesurfer-7.4.1.sif"
OUT = ROOT / "surface_conversion_test_output"
REF_DIR, NIB_DIR, NIB_ORIG_DIR = OUT / "ref", OUT / "nib", OUT / "nib_orig"

SURFACES = ["lh.pial", "rh.pial", "lh.white", "rh.white"]
NAMES = [s.replace(".", "_") for s in SURFACES]
TOL = 1e-4  # max per-coordinate difference [mm]
pd.set_option("display.float_format", "{:.3g}".format)
for d in (REF_DIR, NIB_DIR, NIB_ORIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. FreeSurfer reference

The FreeSurfer image is pulled **once** to a local `.sif`. The build needs ~45 GB of temporary space,
so `APPTAINER_TMPDIR` is pointed at `./apptainer_tmp` (on disk) instead of the tmpfs `/tmp`.
This is the step that fails in the workflow, where every surface triggers a fresh docker→SIF conversion in `/tmp`.

Skipped if all reference STLs already exist; set `FORCE_REFERENCE = True` to regenerate.

In [2]:
FORCE_REFERENCE = False

if FORCE_REFERENCE or not all((REF_DIR / f"{n}.stl").exists() for n in NAMES):
    if not SIF.exists():
        tmp = ROOT / "apptainer_tmp"
        tmp.mkdir(exist_ok=True)
        subprocess.run(
            ["apptainer", "pull", str(SIF), "docker://freesurfer/freesurfer:7.4.1"],
            env={**__import__("os").environ, "APPTAINER_TMPDIR": str(tmp)},
            check=True,
        )
    for surface, name in zip(SURFACES, NAMES):
        subprocess.run(
            [
                "apptainer", "exec", "-e", "--no-mount", "home,cwd", "--cwd", "/",
                "-B", f"{SURF_DIR.resolve()}:/data:ro",
                "-B", f"{REF_DIR.resolve()}:/output",
                "-B", f"{LICENSE.resolve()}:/usr/local/freesurfer/.license:ro",
                str(SIF),
                "mris_convert", "--to-scanner", f"/data/{surface}", f"/output/{name}.stl",
            ],
            check=True,
        )
sorted(p.name for p in REF_DIR.glob("*.stl"))

['lh_pial.stl', 'lh_white.stl', 'rh_pial.stl', 'rh_white.stl']

## 2. nibabel conversion

FreeSurfer surfaces are stored in *tkregister* RAS. `--to-scanner` maps them to scanner RAS with
`vox2ras @ inv(vox2ras_tkr)` of the volume the surface was created from. That geometry is stored in the
surface file's footer (`volume_info`), so no volume is needed. As a cross-check, the same transform is also
computed from the header of `mri/orig.mgz`.

In [3]:
def tkr_to_scanner_from_volume_info(volume_info: dict) -> np.ndarray:
    shape = np.asarray(volume_info["volume"], dtype=float)
    vs = np.asarray(volume_info["voxelsize"], dtype=float)
    mdc = np.column_stack([volume_info["xras"], volume_info["yras"], volume_info["zras"]]).astype(float)
    cras = np.asarray(volume_info["cras"], dtype=float)
    center = shape / 2.0

    vox2ras = np.eye(4)
    vox2ras[:3, :3] = mdc * vs
    vox2ras[:3, 3] = cras - vox2ras[:3, :3] @ center

    # FreeSurfer's tkregister convention (LIA orientation, centred volume).
    vox2ras_tkr = np.eye(4)
    vox2ras_tkr[:3, :3] = [[-vs[0], 0, 0], [0, 0, vs[2]], [0, -vs[1], 0]]
    vox2ras_tkr[:3, 3] = -vox2ras_tkr[:3, :3] @ center
    return vox2ras @ np.linalg.inv(vox2ras_tkr)


def tkr_to_scanner_from_mgh(path: Path) -> np.ndarray:
    header = nib.load(path).header
    return header.get_vox2ras() @ np.linalg.inv(header.get_vox2ras_tkr())


def convert_surface(surface: Path, output: Path, ref_volume: Path | None = None) -> np.ndarray:
    coords, faces, volume_info = nib.freesurfer.read_geometry(surface, read_metadata=True)
    affine = (
        tkr_to_scanner_from_mgh(ref_volume)
        if ref_volume is not None
        else tkr_to_scanner_from_volume_info(volume_info)
    )
    scanner = coords @ affine[:3, :3].T + affine[:3, 3]
    cells = np.column_stack([np.full(len(faces), 3), faces]).ravel()
    pv.PolyData(scanner, cells).save(output, binary=True)
    return affine


affines = {}
for surface, name in zip(SURFACES, NAMES):
    affines[name, "footer"] = convert_surface(SURF_DIR / surface, NIB_DIR / f"{name}.stl")
    affines[name, "orig.mgz"] = convert_surface(SURF_DIR / surface, NIB_ORIG_DIR / f"{name}.stl", ORIG_MGZ)

print("tkr -> scanner (lh.pial, from footer):")
print(np.array2string(affines["lh_pial", "footer"], precision=6, suppress_small=True))
print("max |footer - orig.mgz| over all surfaces:",
      max(np.abs(affines[n, "footer"] - affines[n, "orig.mgz"]).max() for n in NAMES))

tkr -> scanner (lh.pial, from footer):
[[ 1.        0.        0.        0.397308]
 [ 0.        1.        0.       19.349976]
 [ 0.        0.        1.        2.380936]
 [ 0.        0.        0.        1.      ]]
max |footer - orig.mgz| over all surfaces: 0.0


## 3. Geometric comparison

STL is unindexed: each triangle stores its three vertices. Comparing the `(n_triangles, 3, 3)` arrays
directly therefore checks triangle count, triangle order **and** winding at once. `rolled_max` / `flipped_max`
show what the difference would be if the vertex order within each triangle were shifted or reversed. That
they are large confirms the matching order is meaningful. Order-independent checks (Hausdorff distance, bounding
box, area, signed volume) are included as well.

In [4]:
def triangles(mesh: pv.PolyData) -> np.ndarray:
    return np.asarray(mesh.points)[mesh.faces.reshape(-1, 4)[:, 1:]]


def signed_volume(tri: np.ndarray) -> float:
    return float(np.einsum("ij,ij->i", tri[:, 0], np.cross(tri[:, 1], tri[:, 2])).sum() / 6)


def compare(ref_path: Path, cand_path: Path) -> dict:
    ref, cand = pv.read(ref_path), pv.read(cand_path)
    t_ref, t_cand = triangles(ref), triangles(cand)
    res = {"n_tri_ref": len(t_ref), "n_tri_cand": len(t_cand)}
    if t_ref.shape == t_cand.shape:
        diff = np.abs(t_ref - t_cand)
        res |= {
            "tri_max": diff.max(),
            "tri_mean": diff.mean(),
            "rolled_max": np.abs(np.roll(t_ref, 1, axis=1) - t_cand).max(),
            "flipped_max": np.abs(t_ref[:, ::-1] - t_cand).max(),
        }
    else:
        res |= {"tri_max": np.inf, "tri_mean": np.inf}
    p_ref = np.unique(t_ref.reshape(-1, 3), axis=0)
    p_cand = np.unique(t_cand.reshape(-1, 3), axis=0)
    res["hausdorff"] = max(cKDTree(p_cand).query(p_ref)[0].max(), cKDTree(p_ref).query(p_cand)[0].max())
    res["bbox_max"] = np.abs(np.subtract(ref.bounds, cand.bounds)).max()
    res["area_rel"] = abs(ref.area - cand.area) / ref.area
    res["vol_ref"], res["vol_cand"] = signed_volume(t_ref), signed_volume(t_cand)
    res["pass"] = bool(res["tri_max"] <= TOL and np.sign(res["vol_ref"]) == np.sign(res["vol_cand"]))
    return res


geom = pd.DataFrame(
    {n: compare(REF_DIR / f"{n}.stl", NIB_DIR / f"{n}.stl") for n in NAMES}
).T
assert geom["pass"].all(), "nibabel output does not match mris_convert"
geom

,n_tri_ref,n_tri_cand,tri_max,tri_mean,rolled_max,flipped_max,hausdorff,bbox_max,area_rel,vol_ref,vol_cand,pass
lh_pial,213280,213280,0,0,5.71,5.35,0,0,0,4.51e+05,4.51e+05,True
rh_pial,216106,216106,0,0,5.44,5.28,0,0,0,4.57e+05,4.57e+05,True
lh_white,213280,213280,0,0,3.34,3.34,0,0,0,2.48e+05,2.48e+05,True
rh_white,216106,216106,0,0,2.9,2.9,0,0,0,2.5e+05,2.5e+05,True


## 4. Byte-level comparison

A binary STL is:

```
bytes 0-79    header (free text, ignored by readers)
bytes 80-83   uint32 triangle count
per triangle, 50 bytes:
  12 B  normal     (3 x float32)
  36 B  3 vertices (9 x float32)
   2 B  uint16 "attribute byte count" (spec: 0; ignored by readers)
```

The files are not byte-identical. The cells below show that the differences are confined to the header,
the stored normals and the attribute field.

In [5]:
STL_RECORD = np.dtype([("normal", "<f4", 3), ("verts", "<f4", (3, 3)), ("attr", "<u2")])


def read_binary_stl(path: Path) -> tuple[bytes, int, np.ndarray]:
    raw = path.read_bytes()
    n = int.from_bytes(raw[80:84], "little")
    assert len(raw) == 84 + 50 * n, f"{path} is not a binary STL"
    return raw[:80], n, np.frombuffer(raw, STL_RECORD, offset=84)


stl = {(d.name, n): read_binary_stl(d / f"{n}.stl") for d in (REF_DIR, NIB_DIR) for n in NAMES}
for d in ("ref", "nib"):
    print(f"header {d}: {stl[d, 'lh_pial'][0]!r}")

header ref: b'freesurfer mesh\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
header nib: b'Visualization Toolkit generated SLA File\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


### Stored normals

Neither writer is given normals; each recomputes them from the vertices. VTK (via pyvista) computes them from
the float64 coordinates passed in, so they are within float32 rounding of the exact normals. FreeSurfer's are up to ~1e-5
off, consistent with a float32 computation (its normal lengths also deviate more from 1). Normals recomputed by a
reader from the stored float32 vertices deviate *more* (~1e-3 on tiny triangles) than either file's stored normals.

In [6]:
def unit_normals(tri: np.ndarray) -> np.ndarray:
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    return n / np.linalg.norm(n, axis=1, keepdims=True)


rows = {}
for surface, name in zip(SURFACES, NAMES):
    _, n_ref, r = stl["ref", name]
    _, n_nib, c = stl["nib", name]
    coords, faces, vi = nib.freesurfer.read_geometry(SURF_DIR / surface, read_metadata=True)
    exact = unit_normals((coords.astype(np.float64) @ affines[name, "footer"][:3, :3].T
                          + affines[name, "footer"][:3, 3])[faces])
    d = r["normal"].astype(np.float64) - c["normal"]
    eu = np.linalg.norm(d, axis=1)
    rows[name] = {
        "count_equal": n_ref == n_nib,
        "verts_bit_identical": np.array_equal(r["verts"], c["verts"]),
        "normal_max_component": np.abs(d).max(),
        "normal_max_euclid": eu.max(),
        "normal_mean_euclid": eu.mean(),
        "normal_max_angle_deg": np.degrees(eu.max()),
        "normal_bit_identical_%": 100 * np.all(r["normal"] == c["normal"], axis=1).mean(),
        "ref_vs_exact_max": np.abs(r["normal"] - exact).max(),
        "nib_vs_exact_max": np.abs(c["normal"] - exact).max(),
        "ref_len_dev_max": np.abs(np.linalg.norm(r["normal"].astype(float), axis=1) - 1).max(),
        "nib_len_dev_max": np.abs(np.linalg.norm(c["normal"].astype(float), axis=1) - 1).max(),
        "attr_ref": np.unique(r["attr"]).tolist(),
        "attr_nib": np.unique(c["attr"]).tolist(),
    }
bytewise = pd.DataFrame(rows).T
bytewise

,count_equal,verts_bit_identical,normal_max_component,normal_max_euclid,normal_mean_euclid,normal_max_angle_deg,normal_bit_identical_%,ref_vs_exact_max,nib_vs_exact_max,ref_len_dev_max,nib_len_dev_max,attr_ref,attr_nib
lh_pial,True,True,8.79e-06,1.06e-05,5.04e-08,0.000609,11.8,8.78e-06,2.98e-08,1.66e-07,5.05e-08,[48],[0]
rh_pial,True,True,2.94e-06,3.94e-06,5.03e-08,0.000225,11.9,2.94e-06,2.98e-08,1.71e-07,4.89e-08,[48],[0]
lh_white,True,True,1.04e-06,1.21e-06,4.76e-08,6.95e-05,12.9,1.03e-06,2.98e-08,1.59e-07,4.84e-08,[48],[0]
rh_white,True,True,2.15e-06,3.15e-06,4.79e-08,0.00018,12.6,2.14e-06,2.98e-08,1.6e-07,4.89e-08,[48],[0]


### Attribute field and a raw record

FreeSurfer writes `30 00` (= 48 = ASCII `'0'`) in every record's attribute field, VTK writes `00 00` as the spec says.
The first lh_pial record, split into its fields:

In [7]:
def show_record(raw: bytes, i: int = 0) -> str:
    rec = raw[84 + 50 * i: 84 + 50 * (i + 1)]
    words = lambda b: " ".join(b[k:k + 4].hex() for k in range(0, len(b), 4))
    return f"normal {words(rec[:12])} | verts {words(rec[12:48])} | attr {rec[48:].hex()}"


for d in (REF_DIR, NIB_DIR):
    print(f"{d.name:4s}", show_record((d / "lh_pial.stl").read_bytes()))

ref  normal 5cb6b4be 220e543f e9bfdebe | verts a15798c1 aa992942 570b2641 7be399c1 b6f72a42 8ff73241 337b96c1 1cee2a42 1b272d41 | attr 3000
nib  normal 5bb6b4be 220e543f eabfdebe | verts a15798c1 aa992942 570b2641 7be399c1 b6f72a42 8ff73241 337b96c1 1cee2a42 1b272d41 | attr 0000


## 5. Visual inspection (optional)

Visually the two surfaces coincide exactly (identical vertices; viewers recompute normals for shading).
To see *where* the stored normals differ, this writes a `.vtp` per surface with per-triangle cell data:
`normal_diff` (Euclidean norm of the difference), `area`, and both stored normals. Open in ParaView and colour by
`normal_diff`. The largest differences are on very small triangles.

In [8]:
for name in NAMES:
    _, _, r = stl["ref", name]
    _, _, c = stl["nib", name]
    mesh = pv.read(NIB_DIR / f"{name}.stl")
    mesh.cell_data["normal_ref"] = r["normal"]
    mesh.cell_data["normal_nib"] = c["normal"]
    mesh.cell_data["normal_diff"] = np.linalg.norm(r["normal"].astype(np.float64) - c["normal"], axis=1)
    mesh = mesh.compute_cell_sizes(length=False, volume=False)
    mesh.save(OUT / f"{name}_normal_diff.vtp")

worst = int(mesh.cell_data["normal_diff"].argmax())
print(f"written to {OUT}; {name}: worst triangle area {mesh.cell_data['Area'][worst]:.2e} "
      f"mm^2 vs median {np.median(mesh.cell_data['Area']):.2f} mm^2")

written to /home/jorris/project/gonzo/gonzo/surface_conversion_test_output; rh_white: worst triangle area 6.91e-03 mm^2 vs median 0.41 mm^2


## Conclusion

- Vertex coordinates of all four surfaces are **bit-identical** to `mris_convert --to-scanner`, in the same triangle
  order and winding, with the same signed volume.
- The surface-footer transform equals the `orig.mgz` transform (a translation by `c_ras` for conformed volumes).
- Byte differences are confined to fields STL readers ignore: the 80-byte header, the stored normals
  (float32-level differences), and the attribute field (`'0'` vs `0`).
- The nibabel conversion takes seconds and needs neither the FreeSurfer image nor a license.